# 6. System Implementation

This section describes the implementation of the proposed material call decision support system using Python and Pandas.

The implementation is divided into several modules, including data import, data processing, decision generation, and report export.


## 6.1 Import Libraries

The following libraries are required to read Excel files, process data, and generate the recommendation report.

In [1]:
# Import required library

import pandas as pd

## 6.2 Read Input Reports

The system reads all input reports exported from SAP.

The input file contains four worksheets:

- Shortage
- Stock
- Open_PO
- SLOC_PRIORITY

Each worksheet is loaded into a separate Pandas DataFrame for further processing.

In [2]:
# Define input file

file_path = "/kaggle/input/datasets/tanyapornchaichana/input-report/Input_Report.xlsx"

In [3]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/tanyapornchaichana/input-report/Input_Report.xlsx


In [4]:
shortage = pd.read_excel(file_path, sheet_name="shortage")
stock = pd.read_excel(file_path, sheet_name="stock")
open_po = pd.read_excel(file_path, sheet_name="open_po")
SLOC_PRIORITY = pd.read_excel(file_path, sheet_name="SLOC_PRIORITY")

In [5]:
print(open_po.columns.tolist())

['Material', 'ETA', 'Plant', 'Vendor', 'Vendor Available Qty', 'Sloc', 'PO Qty']


In [6]:
# Check whether all worksheets were loaded correctly

print("Shortage shape:", shortage.shape)
print("Stock shape:", stock.shape)
print("Open PO shape:", open_po.shape)
print("SLOC Priority shape:", SLOC_PRIORITY.shape)

print("\nShortage preview:")
display(shortage.head())

print("\nStock preview:")
display(stock.head())

print("\nOpen PO preview:")
display(open_po.head())

print("\nSLOC Priority preview:")
display(SLOC_PRIORITY.head())

Shortage shape: (20, 14)
Stock shape: (18, 4)
Open PO shape: (11, 7)
SLOC Priority shape: (4, 2)

Shortage preview:


,Material,Description,採購員,替代料,OPEN ORDER缺料數,2026-07-06 10:08:19,2026-07-07 10:08:19,2026-07-08 10:08:19,2026-07-09 10:08:19,2026-07-10 10:08:19,2026-07-11 10:08:19,2026-07-12 10:08:19,7Day shortage,ETA
0,60352311231,BATTERY,PIM,Y,385,0,385,0,0,0,0,0,385,NaN
1,60352311232,BATTERY,PIM,Y,8,0,8,0,0,0,0,0,8,NaN
2,60352311233,BATTERY,PIM,Y,7,0,7,0,0,0,0,0,7,NaN
3,60352311234,BATTERY,PIM,Y,255,0,255,0,0,0,0,0,255,NaN
4,60352311235,BATTERY,PIM,Y,14,0,14,0,0,0,0,0,14,NaN



Stock preview:


,Material,Warehouse,Sloc,Qty
0,60352311231,TH05,SW52,50.0
1,60352311232,TH60,NaN,NaN
2,60352311233,TH60,SA01,3430.0
3,60352311234,TH60,NaN,NaN
4,60352311235,TH60,SW01,6500.0



Open PO preview:


,Material,ETA,Plant,Vendor,Vendor Available Qty,Sloc,PO Qty
0,60352311231,NaN,TH05,123123,2000,SW53,1000
1,60352311232,NaN,TH05,123123,1000,SW53,1000
2,60352311233,NaN,TH05,123123,2000,SW53,1000
3,60352311234,NaN,TH05,123123,1000,SW53,1000
4,60352311235,NaN,TH05,123123,2000,SW53,1000



SLOC Priority preview:


,Sloc,Priority
0,IW01,1
1,SW51,2
2,SW01,3
3,SA01,4


## 6.3 Data Inspection

Before processing the data, the input reports are inspected to ensure that they have been loaded correctly.

The inspection includes checking the data size, column names, and sample records.

In [7]:
# Check the size of each DataFrame

print("Shortage:", shortage.shape)
print("Stock:", stock.shape)
print("Open PO:", open_po.shape)
print("SLOC Priority:", SLOC_PRIORITY.shape)

Shortage: (20, 14)
Stock: (18, 4)
Open PO: (11, 7)
SLOC Priority: (4, 2)


In [8]:
print(shortage.columns.tolist())

['Material', 'Description', '採購員', '替代料', 'OPEN ORDER缺料數', datetime.datetime(2026, 7, 6, 10, 8, 19), datetime.datetime(2026, 7, 7, 10, 8, 19), datetime.datetime(2026, 7, 8, 10, 8, 19), datetime.datetime(2026, 7, 9, 10, 8, 19), datetime.datetime(2026, 7, 10, 10, 8, 19), datetime.datetime(2026, 7, 11, 10, 8, 19), datetime.datetime(2026, 7, 12, 10, 8, 19), '7Day shortage', 'ETA']


In [9]:
# Preview data

shortage.head()

,Material,Description,採購員,替代料,OPEN ORDER缺料數,2026-07-06 10:08:19,2026-07-07 10:08:19,2026-07-08 10:08:19,2026-07-09 10:08:19,2026-07-10 10:08:19,2026-07-11 10:08:19,2026-07-12 10:08:19,7Day shortage,ETA
0,60352311231,BATTERY,PIM,Y,385,0,385,0,0,0,0,0,385,NaN
1,60352311232,BATTERY,PIM,Y,8,0,8,0,0,0,0,0,8,NaN
2,60352311233,BATTERY,PIM,Y,7,0,7,0,0,0,0,0,7,NaN
3,60352311234,BATTERY,PIM,Y,255,0,255,0,0,0,0,0,255,NaN
4,60352311235,BATTERY,PIM,Y,14,0,14,0,0,0,0,0,14,NaN


## 6.4 Data Processing

Before generating recommendations, the system summarizes the stock data.

Since the same material may exist in multiple warehouses or SLOCs, the total available quantity must be calculated for each material.

In [10]:
stock.head()

,Material,Warehouse,Sloc,Qty
0,60352311231,TH05,SW52,50.0
1,60352311232,TH60,NaN,NaN
2,60352311233,TH60,SA01,3430.0
3,60352311234,TH60,NaN,NaN
4,60352311235,TH60,SW01,6500.0


In [11]:
print(stock.columns.tolist())

['Material', 'Warehouse', 'Sloc', 'Qty']


In [12]:
available_qty = stock.groupby("Material")["Qty"].sum().reset_index().rename(columns={"Qty": "Available Qty"})
display(available_qty)


,Material,Available Qty
0,60352311231,50.0
1,60352311232,0.0
2,60352311233,3430.0
3,60352311234,0.0
4,60352311235,6500.0
5,60352311236,4712.0
6,60352311237,5802.0
7,60352311238,40829.0
8,60352311239,3301.0
9,60352311240,2521.0


## 6.5 Merge Available Stock

The summarized stock information is merged with the shortage report.

This step allows the system to compare the available quantity against the shortage quantity for each material.

In [13]:
result = shortage.merge(
    available_qty,
    on="Material",
    how="left"
)

display(result.head())

,Material,Description,採購員,替代料,OPEN ORDER缺料數,2026-07-06 10:08:19,2026-07-07 10:08:19,2026-07-08 10:08:19,2026-07-09 10:08:19,2026-07-10 10:08:19,2026-07-11 10:08:19,2026-07-12 10:08:19,7Day shortage,ETA,Available Qty
0,60352311231,BATTERY,PIM,Y,385,0,385,0,0,0,0,0,385,NaN,50.0
1,60352311232,BATTERY,PIM,Y,8,0,8,0,0,0,0,0,8,NaN,0.0
2,60352311233,BATTERY,PIM,Y,7,0,7,0,0,0,0,0,7,NaN,3430.0
3,60352311234,BATTERY,PIM,Y,255,0,255,0,0,0,0,0,255,NaN,0.0
4,60352311235,BATTERY,PIM,Y,14,0,14,0,0,0,0,0,14,NaN,6500.0


## 6.6 Handle Missing Stock

Some materials in the shortage report may not be found in the stock report.

In this case, the missing Available Qty is replaced with 0 before further calculation.

In [14]:
# Replace missing stock values with 0

result["Available Qty"] = result["Available Qty"].fillna(0)

display(result.head())

,Material,Description,採購員,替代料,OPEN ORDER缺料數,2026-07-06 10:08:19,2026-07-07 10:08:19,2026-07-08 10:08:19,2026-07-09 10:08:19,2026-07-10 10:08:19,2026-07-11 10:08:19,2026-07-12 10:08:19,7Day shortage,ETA,Available Qty
0,60352311231,BATTERY,PIM,Y,385,0,385,0,0,0,0,0,385,NaN,50.0
1,60352311232,BATTERY,PIM,Y,8,0,8,0,0,0,0,0,8,NaN,0.0
2,60352311233,BATTERY,PIM,Y,7,0,7,0,0,0,0,0,7,NaN,3430.0
3,60352311234,BATTERY,PIM,Y,255,0,255,0,0,0,0,0,255,NaN,0.0
4,60352311235,BATTERY,PIM,Y,14,0,14,0,0,0,0,0,14,NaN,6500.0


## 6.7 Calculate Stock Balance
The system calculates the stock balance by comparing the available quantity with the 7-day shortage.

Stock Balance = Available Qty - 7Day Shortage

A negative stock balance indicates that additional material is required.

In [15]:
# Calculate stock balance

result["Stock Balance"] = (
    result["Available Qty"] - result["7Day shortage"]
)

display(
    result[
        ["Material", "7Day shortage", "Available Qty", "Stock Balance"]
    ].head()
)

,Material,7Day shortage,Available Qty,Stock Balance
0,60352311231,385,50.0,-335.0
1,60352311232,8,0.0,-8.0
2,60352311233,7,3430.0,3423.0
3,60352311234,255,0.0,-255.0
4,60352311235,14,6500.0,6486.0


## 6.8 Calculate Call Quantity

If the stock balance is negative, the system converts the shortage amount into a positive call quantity.

Call Qty = ABS(Stock Balance)

The Call Qty represents the quantity that must be obtained from the vendor.

In [16]:
# Calculate required call quantity

result["Call Qty"] = (
    -result["Stock Balance"]
).clip(lower=0)

display(
    result[
        [
            "Material",
            "7Day shortage",
            "Available Qty",
            "Stock Balance",
            "Call Qty"
        ]
    ].head()
)

,Material,7Day shortage,Available Qty,Stock Balance,Call Qty
0,60352311231,385,50.0,-335.0,335.0
1,60352311232,8,0.0,-8.0,8.0
2,60352311233,7,3430.0,3423.0,0.0
3,60352311234,255,0.0,-255.0,255.0
4,60352311235,14,6500.0,6486.0,0.0


## 6.9 Merge Vendor and ETA Information

The system merges vendor available quantity, purchase order quantity, and ETA information from the Open_PO report.

This information is used to evaluate whether the required call quantity can be fulfilled by the vendor or whether the incoming material must be expedited.

In [17]:
vendor_info = open_po[
    [
        "Material",
        "Vendor",
        "Vendor Available Qty",
        "PO Qty"
    ]
]

result = result.merge(
    vendor_info,
    on="Material",
    how="left"
)

display(result.head())

,Material,Description,採購員,替代料,OPEN ORDER缺料數,2026-07-06 10:08:19,2026-07-07 10:08:19,2026-07-08 10:08:19,2026-07-09 10:08:19,2026-07-10 10:08:19,2026-07-11 10:08:19,2026-07-12 10:08:19,7Day shortage,ETA,Available Qty,Stock Balance,Call Qty,Vendor,Vendor Available Qty,PO Qty
0,60352311231,BATTERY,PIM,Y,385,0,385,0,0,0,0,0,385,NaN,50.0,-335.0,335.0,123123.0,2000.0,1000.0
1,60352311232,BATTERY,PIM,Y,8,0,8,0,0,0,0,0,8,NaN,0.0,-8.0,8.0,123123.0,1000.0,1000.0
2,60352311233,BATTERY,PIM,Y,7,0,7,0,0,0,0,0,7,NaN,3430.0,3423.0,0.0,123123.0,2000.0,1000.0
3,60352311234,BATTERY,PIM,Y,255,0,255,0,0,0,0,0,255,NaN,0.0,-255.0,255.0,123123.0,1000.0,1000.0
4,60352311235,BATTERY,PIM,Y,14,0,14,0,0,0,0,0,14,NaN,6500.0,6486.0,0.0,123123.0,2000.0,1000.0


## 6.10 Check Vendor Available Stock

The system checks the vendor information after merging the Open_PO report.

This step verifies that the vendor name, vendor available quantity, and purchase order quantity have been loaded correctly before generating the final decision.

In [18]:
# Check vendor information

display(
    result[
        [
            "Material",
            "Call Qty",
            "Vendor",
            "Vendor Available Qty",
            "PO Qty"
        ]
    ].head()
)

,Material,Call Qty,Vendor,Vendor Available Qty,PO Qty
0,60352311231,335.0,123123.0,2000.0,1000.0
1,60352311232,8.0,123123.0,1000.0,1000.0
2,60352311233,0.0,123123.0,2000.0,1000.0
3,60352311234,255.0,123123.0,1000.0,1000.0
4,60352311235,0.0,123123.0,2000.0,1000.0


## 6.11 Generate Decision

The system generates a decision for each material based on the stock balance, vendor available quantity, and ETA.

Possible decisions:

- Transfer Stock
- CALL Material
- Wait for Material Arrival
- PLS PULL IN ETA

In [19]:
# Generate decision for each material

def generate_decision(row):

    if row["Stock Balance"] >= 0:
        return "Transfer Stock"

    elif row["Vendor Available Qty"] >= row["Call Qty"]:
        return "CALL Material"

    elif pd.notna(row["ETA"]):
        return "Wait for Material Arrival"

    else:
        return "PLS PULL IN ETA"


result["Decision"] = result.apply(
    generate_decision,
    axis=1
)

display(
    result[
        [
            "Material",
            "Stock Balance",
            "Call Qty",
            "Vendor Available Qty",
            "ETA",
            "Decision"
        ]
    ].head()
)

,Material,Stock Balance,Call Qty,Vendor Available Qty,ETA,Decision
0,60352311231,-335.0,335.0,2000.0,NaN,CALL Material
1,60352311232,-8.0,8.0,1000.0,NaN,CALL Material
2,60352311233,3423.0,0.0,2000.0,NaN,Transfer Stock
3,60352311234,-255.0,255.0,1000.0,NaN,CALL Material
4,60352311235,6486.0,0.0,2000.0,NaN,Transfer Stock


## 6.12 Final Recommendation

The final recommendation report summarizes the stock analysis and purchasing decision for each material.

The report includes shortage quantity, available stock, stock balance, call quantity, vendor information, ETA, and the final recommendation.

In [20]:
# Create final recommendation report

final_report = result[
    [
        "Material",
        "7Day shortage",
        "Available Qty",
        "Stock Balance",
        "Call Qty",
        "Vendor",
        "Vendor Available Qty",
        "PO Qty",
        "ETA",
        "Decision"
    ]
]

display(final_report)

,Material,7Day shortage,Available Qty,Stock Balance,Call Qty,Vendor,Vendor Available Qty,PO Qty,ETA,Decision
0,60352311231,385,50.0,-335.0,335.0,123123.0,2000.0,1000.0,NaN,CALL Material
1,60352311232,8,0.0,-8.0,8.0,123123.0,1000.0,1000.0,NaN,CALL Material
2,60352311233,7,3430.0,3423.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
3,60352311234,255,0.0,-255.0,255.0,123123.0,1000.0,1000.0,NaN,CALL Material
4,60352311235,14,6500.0,6486.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
5,60352311236,14,4712.0,4698.0,0.0,123123.0,1000.0,1000.0,NaN,Transfer Stock
6,60352311237,46,5802.0,5756.0,0.0,231231.0,2000.0,3000.0,NaN,Transfer Stock
7,60352311238,1168,40829.0,39661.0,0.0,231231.0,1000.0,3000.0,NaN,Transfer Stock
8,60352311239,14,3301.0,3287.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
9,60352311240,1,2521.0,2520.0,0.0,123123.0,1000.0,2000.0,7/6*1,Transfer Stock


### 6.13 Export Result

The final recommendation report is exported to an Excel file for purchasing planning and decision making.

In [21]:
# Export the final recommendation report

final_report.to_excel(
    "Material_Call_Recommendation.xlsx",
    index=False
)

print("Export completed successfully.")

Export completed successfully.


# 7. Results and Discussion

The system successfully combines shortage, stock, vendor, and ETA information into one recommendation report.

For each material, the system calculates the available quantity, stock balance, and call quantity before generating a final decision.

The possible decisions are:

- Transfer Stock
- CALL Material
- Wait for Material Arrival
- PLS PULL IN ETA

The output helps material planners review shortages more efficiently and reduces manual checking across multiple reports.

## 7.1 Sample Output

In [22]:
display(final_report.head(10))

,Material,7Day shortage,Available Qty,Stock Balance,Call Qty,Vendor,Vendor Available Qty,PO Qty,ETA,Decision
0,60352311231,385,50.0,-335.0,335.0,123123.0,2000.0,1000.0,NaN,CALL Material
1,60352311232,8,0.0,-8.0,8.0,123123.0,1000.0,1000.0,NaN,CALL Material
2,60352311233,7,3430.0,3423.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
3,60352311234,255,0.0,-255.0,255.0,123123.0,1000.0,1000.0,NaN,CALL Material
4,60352311235,14,6500.0,6486.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
5,60352311236,14,4712.0,4698.0,0.0,123123.0,1000.0,1000.0,NaN,Transfer Stock
6,60352311237,46,5802.0,5756.0,0.0,231231.0,2000.0,3000.0,NaN,Transfer Stock
7,60352311238,1168,40829.0,39661.0,0.0,231231.0,1000.0,3000.0,NaN,Transfer Stock
8,60352311239,14,3301.0,3287.0,0.0,123123.0,2000.0,1000.0,NaN,Transfer Stock
9,60352311240,1,2521.0,2520.0,0.0,123123.0,1000.0,2000.0,7/6*1,Transfer Stock


## 7.2 Result Interpretation
- Transfer Stock: Internal stock is sufficient to cover the shortage.
- CALL Material: Vendor available quantity is sufficient for the required call quantity.
- Wait for Material Arrival: Vendor stock is insufficient, but an ETA is available.
- PLS PULL IN ETA: No sufficient stock or confirmed ETA is available, so delivery must be expedited.